# BigAlpha F1 Tier 1 inner-selected epoch Soft-Spearman ensemble

This submission uses all 26 numeric raw fields from the permitted 28-column
one-minute schema; date and instrument remain keys. It loads three fixed-epoch
causal multi-scale models
trained with seeds 11, 29, and 47 and averages their within-date percentile
ranks.

The public stage loads `weights.json`. The private stage imports
`train_and_save` to reproduce the fixed 2022-2023, 15-epoch training
configuration. Inference is instrument-chunked to bound memory.


In [ ]:
import os

import dai
import numpy as np
import pandas as pd
import structlog
import torch
from train import (
    MODEL_PATH,
    load_ensemble,
    pool,
    predict_scores,
    train_and_save,
)

logger = structlog.get_logger()


def main(datasources, start_date, end_date):
    """Return exact competition columns: date, instrument, score."""
    table = datasources["bar1m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Missing {MODEL_PATH}; upload weights.json with this notebook."
        )

    checkpoint, models = load_ensemble(MODEL_PATH, map_location=device)
    stats = (
        np.asarray(checkpoint["mean"], np.float32),
        np.asarray(checkpoint["std"], np.float32),
    )
    logger.info(
        "ensemble loaded",
        models=len(models),
        path=MODEL_PATH,
        device=str(device),
    )
    predictions = predict_scores(
        models,
        table,
        start_date,
        end_date,
        pool(start_date, end_date),
        stats,
        device,
    )

    official = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    result = (
        pd.merge(predictions, official, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])
        [["date", "instrument", "score"]]
        .reset_index(drop=True)
    )
    logger.info(
        "scores complete",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
    )
    return result


if __name__ == "__main__":
    from bigmodule import M

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    if not os.path.exists(MODEL_PATH):
        train_and_save(datasources)
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"
    score_data = main(datasources, start_date, end_date)
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
